In [19]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
!pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\dev\python.exe -m pip install --upgrade pip


In [21]:
import os
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.chains import RetrievalQA






In [22]:
df = pd.read_csv("ecommerce_products_dataset.csv")

In [23]:
documents = []
for _, row in df.iterrows():
    content = f"""
Product ID: {row['product_id']}
Product Name: {row['product_name']}
Category: {row['category']}
Brand: {row['brand']}
Price: {row['price']}
Rating: {row['rating']}
Stock: {row['stock']}
Discount Percent: {row['discount_percent']}
Seller: {row['seller']}
Description: {row['description']}
"""
    documents.append(
        Document(
            page_content=content,
            metadata={
                "product_id": row["product_id"],
                "product_name": row["product_name"],
                "category": row["category"],
                "brand": row["brand"]
            }
        )
    )


In [24]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(documents)

In [25]:
embeddings = OpenAIEmbeddings()

ecommerce_db = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory="ecommerce_rag_db"
)
ecommerce_db.persist()




In [26]:
retriever = ecommerce_db.as_retriever()

results = retriever.invoke(
    "Show premium electronics with high ratings"
)

for doc in results:
    print(doc.metadata)

{'category': 'Home', 'product_id': 'P10184', 'brand': 'Prestige', 'product_name': 'Prestige LED Table Lamp'}
{'product_name': 'Prestige LED Table Lamp', 'product_id': 'P10184', 'brand': 'Prestige', 'category': 'Home'}
{'product_id': 'P10184', 'category': 'Home', 'product_name': 'Prestige LED Table Lamp', 'brand': 'Prestige'}
{'product_name': 'Prestige LED Table Lamp', 'category': 'Home', 'brand': 'Prestige', 'product_id': 'P10184'}


In [27]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=llm_based_retriever
)
query = "Recommend premium electronics with high ratings"
result = qa_chain({"query": query})

print(result["result"])

 Sony Bluetooth Speaker from PrimeSeller would be a good option as it is a premium product with high ratings and long battery life.


In [28]:
query = "Recommend low budget electronics with high ratings"
result = qa_chain({"query": query})

print(result["result"])


One option could be the Samsung Power Bank from TrustedStore, as it is described as budget-friendly and has AI-ready technology. Another option could be the Logitech Wireless Bluetooth Headphones from TrustedStore, as it also has a budget-friendly price and a high rating of 3.8.


In [29]:
llm = OpenAI(temperature=0)

llm_based_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

docs = llm_based_retriever.invoke(
    "Suggest budget‑friendly electronics with good ratings"
)

for d in docs:
    print(d.metadata)

{'brand': 'Samsung', 'product_id': 'P10128', 'category': 'Electronics', 'product_name': 'Samsung Power Bank'}
{'product_id': 'P10158', 'brand': 'JBL', 'category': 'Electronics', 'product_name': 'JBL Gaming Mouse'}


In [30]:
def build_product_text(df):
    df = df.fillna("").astype(str)
    df["product_text"] = (
        "Product Name: " + df["product_name"] + ". "
        "Category: " + df["category"] + ". "
        "Brand: " + df["brand"] + ". "
        "Price: " + df["price"] + ". "
        "Rating: " + df["rating"] + ". "
        "Description: " + df["description"]
    )
    return df

def simple_retrieve(df, query, top_k=5):
    query_terms = query.lower().split()
    matches = []

    for _, row in df.iterrows():
        text = row["product_text"].lower()
        score = sum(1 for t in query_terms if t in text)
        if score > 0:
            matches.append((score, row["product_name"], row["rating"]))

    matches.sort(reverse=True)
    return matches[:top_k]

df = build_product_text(df)
simple_retrieve(df, "premium electronics high rating")

[(3, 'Sony Webcam', '4.8'),
 (3, 'Sony USB-C Fast Charger', '3.7'),
 (3, 'Sony Smartphone Case', '3.8'),
 (3, 'Sony Power Bank', '3.6'),
 (3, 'Sony Bluetooth Speaker', '4.5')]